In [1]:
import os
from datetime import datetime
import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import lightning.pytorch as pl

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f'Training on device {device}\n')

Training on device cuda



In [3]:
def get_lat_weights(latitude_values):
    lat_rad = np.deg2rad(latitude_values)
    weights = np.cos(lat_rad)
    return weights / np.mean(weights)

In [4]:
class Normalizer:
    def __init__(self):
        self.mean_in, self.std_in = None, None
        self.mean_out, self.std_out = None, None

    def set_input_statistics(self, mean, std):
        self.mean_in = mean
        self.std_in = std

    def set_output_statistics(self, mean, std):
        self.mean_out = mean
        self.std_out = std

    def normalize(self, data, data_type):
        if data_type == "input":
            return (data - self.mean_in) / self.std_in
        elif data_type == "output":
            return (data - self.mean_out) / self.std_out

    def inverse_transform_output(self, data):
        return data * self.std_out + self.mean_out

In [5]:
class ClimateDataset(Dataset):
    def __init__(self, inputs_dask, outputs_dask, output_is_normalized=True):
        self.size = inputs_dask.shape[0]
        print(f"Creating dataset with {self.size} samples...")

        inputs_np = inputs_dask.compute()
        outputs_np = outputs_dask.compute()

        self.inputs = torch.from_numpy(inputs_np).float()
        self.outputs = torch.from_numpy(outputs_np).float()

        if torch.isnan(self.inputs).any() or torch.isnan(self.outputs).any():
            raise ValueError("NaNs found in dataset")

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.inputs[idx], self.outputs[idx]

In [6]:
path = "../processed_data_cse151b_v2_corrupted_ssp245/processed_data_cse151b_v2_corrupted_ssp245.zarr"
input_vars = ["CO2", "SO2", "CH4", "BC", "rsdt"]
output_vars = ["tas", "pr"]
train_ssps = ["ssp126", "ssp370", "ssp585"]
test_ssp = "ssp245"
target_member_id = 0
val_split = 0.1
test_months = 360
batch_size = 32
num_workers = 0
seed = 42
normalizer = Normalizer()

In [7]:
ds = xr.open_zarr(path, consolidated=False, chunks={"time": 24})
spatial_template = ds["rsdt"].isel(time=0, ssp=0, drop=True)
lat = spatial_template.y.values
lon = spatial_template.x.values
area_weights = xr.DataArray(get_lat_weights(lat), dims=["y"], coords={"y": lat})

ds

<xarray.Dataset> Size: 621MB
Dimensions:    (member_id: 3, x: 72, y: 48, longitude: 72, ssp: 4, time: 1021,
                latitude: 48)
Coordinates:
  * member_id  (member_id) int64 24B 0 1 2
    lat        (x, y) float64 28kB dask.array<chunksize=(72, 48), meta=np.ndarray>
    lon        (x, y) float64 28kB dask.array<chunksize=(72, 48), meta=np.ndarray>
  * y          (y) float64 384B -88.59 -84.82 -81.05 ... 81.05 84.82 88.59
  * longitude  (longitude) float64 576B 1.25 6.25 11.25 ... 346.2 351.2 356.2
  * latitude   (latitude) float64 384B -89.05 -85.26 -81.47 ... 85.26 89.05
  * time       (time) object 8kB 2015-01-15 00:00:00 ... 2100-01-15 00:00:00
  * x          (x) float64 576B 1.875 6.875 11.88 16.88 ... 346.9 351.9 356.9
  * ssp        (ssp) <U6 96B 'ssp126' 'ssp245' 'ssp370' 'ssp585'
Data variables:
    CO2        (ssp, time) float64 33kB dask.array<chunksize=(1, 24), meta=np.ndarray>
    pr         (ssp, time, member_id, y, x) float32 169MB dask.array<chunksize=(1, 24, 1, 48, 72), meta=np.ndarray>
    tas        (ssp, time, member_id, y, x) float32 169MB dask.array<chunksize=(1, 24, 1, 48, 72), meta=np.ndarray>
    CH4        (ssp, time) float64 33kB dask.array<chunksize=(1, 24), meta=np.ndarray>
    BC         (ssp, time, latitude, longitude) float64 113MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
    rsdt       (ssp, time, y, x) float32 56MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
    SO2        (ssp, time, latitude, longitude) float64 113MB dask.array<chunksize=(1, 24, 48, 72), meta=np.ndarray>
Attributes:
    original_member_ids:  ['r10i1p1f1', 'r11i1p1f1', 'r4i1p1f1']
    precipitation_units:  mm/day (converted from kg m-2 s-1)
    source:               CMIP6 data processed for CSE151B
    ssp:                  ssp126

In [8]:
def load_ssp(ssp, output):
    input_dask, output_dask = [], []
    for var in input_vars:
        da_var = ds[var].sel(ssp=ssp)
        if "latitude" in da_var.dims:
            da_var = da_var.rename({"latitude": "y", "longitude": "x"})
        if "member_id" in da_var.dims:
            da_var = da_var.sel(member_id=target_member_id)
        if set(da_var.dims) == {"time"}:
            da_var = da_var.broadcast_like(spatial_template).transpose("time", "y", "x")
        input_dask.append(da_var.data)

    da_out = ds[output].sel(ssp=ssp, member_id=target_member_id)
    if "latitude" in da_out.dims:
        da_out = da_out.rename({"latitude": "y", "longitude": "x"})
    output_dask.append(da_out.data)

    return da.stack(input_dask, axis=1), da.stack(output_dask, axis=1)

In [9]:
def create_sequences(x, y, seq_len, pad_value=0.0):
    X_seq, Y_seq = [], []
    T = len(x)
    C, H, W = x.shape[1:]

    for t in range(T):
        start_idx = max(0, t - seq_len + 1)
        actual_seq = x[start_idx:t+1]

        padded_seq = np.full((seq_len, C, H, W), fill_value=pad_value, dtype=x.dtype)
        padded_seq[-(t - start_idx + 1):] = actual_seq

        X_seq.append(padded_seq)
        Y_seq.append(y[t])

    return X_seq, Y_seq


In [10]:
def preprocess(output, normalizer):
    raw_train_inputs = []
    raw_train_outputs = []
    
    ssp_dc = {}
    
    for ssp in train_ssps:
        x, y = load_ssp(ssp, output=output)
    
        if ssp == "ssp370":
            ssp_dc[ssp] = (x[:-test_months], y[:-test_months])
        else:
            ssp_dc[ssp] = (x, y)
    
        raw_train_inputs.append(ssp_dc[ssp][0])
        raw_train_outputs.append(ssp_dc[ssp][1])
    
    raw_train_inputs = da.concatenate(raw_train_inputs, axis=0)
    raw_train_outputs = da.concatenate(raw_train_outputs, axis=0)
    
    normalizer.set_input_statistics(
        mean=da.nanmean(raw_train_inputs, axis=(0, 2, 3), keepdims=True).compute(),
        std=da.nanstd(raw_train_inputs, axis=(0, 2, 3), keepdims=True).compute(),
    )
    normalizer.set_output_statistics(
        mean=da.nanmean(raw_train_outputs, axis=(0, 2, 3), keepdims=True).compute(),
        std=da.nanstd(raw_train_outputs, axis=(0, 2, 3), keepdims=True).compute(),
    )

    train_input, train_output = [], []
    val_input, val_output = [], []
    
    for ssp in train_ssps:
        x_train = normalizer.normalize(ssp_dc[ssp][0], 'input')
        y_train = normalizer.normalize(ssp_dc[ssp][1], 'output')
        if ssp == "ssp370":
            x_val = normalizer.normalize(x[-test_months:], 'input')
            y_val = normalizer.normalize(y[-test_months:], 'output')
            X_val, Y_val = create_sequences(x_val, y_val, seq_len=12)
            
            val_input.extend(X_val)
            val_output.extend(Y_val)
        
        X_seq, Y_seq = create_sequences(x_train, y_train, seq_len=12)
        train_input.extend(X_seq)
        train_output.extend(Y_seq)

    train_input = da.stack(train_input)
    train_output = da.stack(train_output)
    
    val_input = da.stack(val_input)
    val_output = da.stack(val_output)

    raw_test_input, raw_test_output = load_ssp(test_ssp, output=output)
    raw_test_input = raw_test_input[-test_months:]
    raw_test_output = raw_test_output[-test_months:]
    x_test = normalizer.normalize(raw_test_input, "input")
    X_test, Y_test = create_sequences(x_test, raw_test_output, seq_len=12)
    
    test_input = da.stack(X_test)
    test_output = da.stack(Y_test)

    print(val_input.shape, val_output.shape)

    return {'train_in': train_input, 'train_out': train_output, 'val_in': val_input, 'val_out': val_output, 'test_in': test_input, 'test_out': test_output}
    

In [11]:
ta_normalizer = Normalizer()
pr_normalizer = Normalizer()

# ttv_split_ta = preprocess(output='tas', normalizer=ta_normalizer)
# print('Preprocess tas complete')

ttv_split_pr = preprocess(output='pr', normalizer=pr_normalizer)
print('Preprocess pr complete')

# train_dataset_ta = ClimateDataset(ttv_split_ta['train_in'], ttv_split_ta['train_out'])
# val_dataset_ta = ClimateDataset(ttv_split_ta['val_in'], ttv_split_ta['val_out'])
# test_dataset_ta = ClimateDataset(ttv_split_ta['test_in'], ttv_split_ta['test_out'], output_is_normalized=False)

train_dataset_pr = ClimateDataset(ttv_split_pr['train_in'], ttv_split_pr['train_out'])
val_dataset_pr = ClimateDataset(ttv_split_pr['val_in'], ttv_split_pr['val_out'])
test_dataset_pr = ClimateDataset(ttv_split_pr['test_in'], ttv_split_pr['test_out'], output_is_normalized=False)

# train_loader_ta = DataLoader(train_dataset_ta, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
# val_loader_ta = DataLoader(val_dataset_ta, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
# test_dataloader_ta = DataLoader(test_dataset_ta, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

train_loader_pr = DataLoader(train_dataset_pr, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader_pr = DataLoader(val_dataset_pr, batch_size=batch_size, shuffle=True,num_workers=num_workers, pin_memory=True)
test_dataloader_pr = DataLoader(test_dataset_pr, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

(360, 12, 5, 48, 72) (360, 1, 48, 72)
Preprocess pr complete
Creating dataset with 2703 samples...
Creating dataset with 360 samples...
Creating dataset with 360 samples...


In [12]:
class CNNEncoder(nn.Module):
    def __init__(self, in_channels, embed_dim, input_height, input_width, dropout_rate=0.5):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            # nn.Dropout2d(0.1),
            # nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            # nn.Dropout2d(0.2), 
            # nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            # nn.Dropout2d(0.3), 
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            # nn.Dropout2d(0.3),
        )
        H_out, W_out = input_height, input_width 
        self.fc = nn.Sequential(
            nn.Linear(256 * H_out * W_out, embed_dim),
            # nn.Dropout(dropout_rate) 
        )
    
    def forward(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

In [13]:
class RNN(nn.Module):
    def __init__(self, embed_dim, hidden_dim, output_dim):
        super().__init__()
        self.rnn = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x_seq):
        out, _ = self.rnn(x_seq)
        out = out[:, -1]
        return self.fc(out)   

In [14]:
class Transformer(nn.Module):
    def __init__(self, embed_dim, hidden_dim, output_dim, num_heads=4, num_layers=2, dropout=0):
        super().__init__()

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(embed_dim, output_dim)

    def forward(self, x_seq):
        out = self.transformer(x_seq)
        out = out[:, -1]     
        return self.fc(out)

In [15]:
class SpatiotemporalModel(nn.Module):
    def __init__(self, in_channels, out_channels, embed_dim, hidden_dim, H, W, temporal_model):
        super().__init__()
        self.encoder = CNNEncoder(in_channels, embed_dim, H,W)
        self.temporal = temporal_model(embed_dim, hidden_dim, out_channels * H * W)
        
        self.H = H
        self.W = W
        self.out_channels = out_channels
        self.in_channels = in_channels

    def forward(self, x):
        B, T, C, H, W = x.shape
        x_seq = []
        for t in range(T):
            x_t = self.encoder(x[:, t])
            x_seq.append(x_t)
        x_seq = torch.stack(x_seq, dim=1)
        
        out = self.temporal(x_seq)
        
        return out.view(B, self.out_channels, H, W) 
        

In [16]:
def train(model, optimizer, criterion, train_loader, val_loader):
    for epoch in range(15):
        model.train()
        total_train_loss = 0
    
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
    
            pred = model(xb)
            loss = criterion(pred, yb)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
            total_train_loss += loss.item()
    
        model.eval()
    
        with torch.no_grad():
            all_preds = []
            all_targets = []
            total_val_loss = 0
            
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
    
                pred = model(xb)
                loss = criterion(pred, yb)
                total_val_loss += loss.item()
    
                all_preds.append(pred.cpu())
                all_targets.append(yb.cpu())
            
            all_preds = torch.cat(all_preds, dim=0)
            all_targets = torch.cat(all_targets, dim=0)
    
            all_preds = normalizer.inverse_transform_output(all_preds.detach().cpu().numpy())
            all_targets = normalizer.inverse_transform_output(all_targets.detach().cpu().numpy())
            
            time = np.arange(all_preds.shape[0])
    
            print(f"Epoch {epoch+1}: Train Loss = {total_train_loss / len(train_loader):.4f}, Valid Loss = {total_val_loss / len(val_loader):.4f}")  
        
            for i, var in enumerate(['pr']):
                p = all_preds[:, i]
                t = all_targets[:, i]
        
                p_xr = xr.DataArray(p, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
                t_xr = xr.DataArray(t, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
                
                rmse = np.sqrt(((p_xr - t_xr) ** 2).weighted(area_weights).mean(("time", "y", "x")).item())
                mean_rmse = np.sqrt(((p_xr.mean("time") - t_xr.mean("time")) ** 2).weighted(area_weights).mean(("y", "x")).item())
                std_mae = np.abs(p_xr.std("time") - t_xr.std("time")).weighted(area_weights).mean(("y", "x")).item()
        
                print(f"{var}: RMSE={rmse:.4f}, Time-Mean RMSE={mean_rmse:.4f}, Time-Stddev MAE={std_mae:.4f}")
    
        print('\n')

In [17]:
model = SpatiotemporalModel(in_channels=5, out_channels=1, embed_dim=256, hidden_dim=1024, H=48, W=72, temporal_model=RNN).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

train(model, optimizer, criterion, train_loader_pr, val_loader_pr)

TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'

In [ ]:
model.eval()

all_preds = []
all_targets = []

val_loss = 0

with torch.no_grad():
    for xb, yb in val_loader:
        xb, yb = xb.to(device), yb.to(device)

        pred = model(xb)
        loss = criterion(pred, yb)

        val_loss += loss.item()

        all_preds.append(pred.cpu())
        all_targets.append(yb.cpu())

print(f'Final Normalized Validation Loss: {val_loss / len(val_loader):.4f}')

all_preds = torch.cat(all_preds, dim=0)
all_targets = torch.cat(all_targets, dim=0)

all_preds = normalizer.inverse_transform_output(all_preds.detach().cpu().numpy())
all_targets = normalizer.inverse_transform_output(all_targets.detach().cpu().numpy())

for i, var in enumerate(output_vars):
    p = all_preds[:, i]
    t = all_targets[:, i]

    p_xr = xr.DataArray(p, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
    t_xr = xr.DataArray(t, dims=["time", "y", "x"], coords={"time": time, "y": lat, "x": lon})
    
    rmse = np.sqrt(((p_xr - t_xr) ** 2).weighted(area_weights).mean(("time", "y", "x")).item())
    mean_rmse = np.sqrt(((p_xr.mean("time") - t_xr.mean("time")) ** 2).weighted(area_weights).mean(("y", "x")).item())
    std_mae = np.abs(p_xr.std("time") - t_xr.std("time")).weighted(area_weights).mean(("y", "x")).item()

    print(f"{var}: RMSE={rmse:.4f}, Time-Mean RMSE={mean_rmse:.4f}, Time-Stddev MAE={std_mae:.4f}")